# Creating Figures of the Durable Model

Uses `DurableConsumer` from `DurableModel.py` with the four pedagogical channels:
1. **EV Type I taste only** (`sigma_eps > 0`, `SearchCost = 0`, `PriceShkStd = 0`)
2. **Search only** (`SearchCost > 0`, `sigma_eps = 0`, `PriceShkStd = 0`)
3. **Price only** (`PriceShkStd > 0`, hard max / `sigma_eps = 0`, `SearchCost = 0`)
4. **Search + price** (Combined nest; `sigma_eps = 0`)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import time
import datetime
from pathlib import Path

start_time = time.time()

from DurableModel import DurableConsumer
from HARK.ConsumptionSaving.ConsIndShockModel import utility_inv

FIG_DIR = Path("../Figures/DurableModel")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Ordered sweeps, low to high. logs/20261015_Figures.txt.
SWEEP_COLORS = ("#000000", "#0072B2", "#D55E00", "#009E73")
SWEEP_LW = 1.6
N_SHOW_DEFAULT = (0.5, 1.0, 1.5, 2.0)
_KEEP_TINT = (240 / 255, 240 / 255, 240 / 255)
_ADJ_TINT = (214 / 255, 230 / 255, 245 / 255)


def _legend_under_title(ax, handles, labels):
    """Legend row under the axes title, above the plot, frame off."""
    if not handles:
        return
    ax.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(labels),
        frameon=False,
        borderaxespad=0.25,
    )
    title = ax.get_title()
    if title:
        ax.set_title(title, pad=20)


def _legend_under_suptitle(fig, handles, labels):
    """One legend for the figure, under the suptitle."""
    if not handles:
        return
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=len(labels),
        frameon=False,
        borderaxespad=0.0,
    )


def channel_title(sigma_eps, SearchCost, PriceShkStd, extra=None):
    """Parameter header in the style of Figures.ipynb."""
    s = (
        rf"$\sigma_{{\zeta}} = {sigma_eps}$, "
        rf"$S = {SearchCost}$, "
        rf"$\sigma_{{p}} = {PriceShkStd}$"
    )
    if extra:
        s = s + ", " + extra
    return s


def _plot_adjuster_quantity(
    models,
    labels,
    t,
    title,
    fname,
    single_attr,
    list_attr,
    ylabel,
    x_max=None,
    price_nodes=False,
    derivative=False,
):
    """Plot one adjuster quantity, optionally showing every price-node function."""
    if not price_nodes:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for model, lab, color in zip(models, labels, SWEEP_COLORS):
            sol = model.solution[t]
            x = np.asarray(model.xNrmGrid, dtype=float)
            if x_max is not None:
                x = x[x <= x_max]
            y = np.asarray(getattr(sol, single_attr)(x), dtype=float)
            if derivative:
                y = np.gradient(y, x)
            ax.plot(x, y, color=color, linestyle="-", linewidth=SWEEP_LW, label=lab)
        ax.set_xlabel(r"Liquidity, $\ell_t$")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        handles, labs = ax.get_legend_handles_labels()
        _legend_under_title(ax, handles, labs)
        fig.tight_layout()
        fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()
        return

    # One subplot per parameterization; each subplot contains all price nodes.
    n = len(models)
    if n == 1:
        fig, axes = plt.subplots(1, 1, figsize=(6.5, 4.5))
        axes = np.array([axes])
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()

    all_prices = np.concatenate(
        [np.asarray(model.PriceShkVals, dtype=float) for model in models]
    )
    p_min, p_max = float(all_prices.min()), float(all_prices.max())
    if np.isclose(p_min, p_max):
        p_min, p_max = p_min - 0.01, p_max + 0.01
    norm = plt.Normalize(vmin=p_min, vmax=p_max)
    cmap = plt.get_cmap("viridis")

    for ax, model, lab in zip(axes, models, labels):
        sol = model.solution[t]
        funcs = getattr(sol, list_attr, None)
        if funcs is None:
            raise RuntimeError(
                "Price-node adjuster functions are unavailable. Reload "
                "DurableModel.py and solve the models again."
            )
        x = np.asarray(model.xNrmGrid, dtype=float)
        if x_max is not None:
            x = x[x <= x_max]
        for p0, func in zip(model.PriceShkVals, funcs):
            y = np.asarray(func(x), dtype=float)
            if derivative:
                y = np.gradient(y, x)
            ax.plot(x, y, color=cmap(norm(float(p0))), linewidth=1.1)
        ax.set_title(lab)
        ax.set_xlabel(r"Liquidity, $\ell_t$")
        ax.set_ylabel(ylabel)

    for ax in axes[n:]:
        ax.set_visible(False)

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    visible_axes = list(axes[:n])
    fig.colorbar(
        sm,
        ax=visible_axes,
        orientation="horizontal",
        fraction=0.045,
        pad=0.08,
        label=r"Price realization, $p$",
    )
    fig.suptitle(title, y=0.98)
    fig.subplots_adjust(left=0.09, right=0.98, bottom=0.15, top=0.90, hspace=0.28)
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


def plot_adjuster_policies(
    models, labels, t, title, fname_prefix, x_max=None, price_nodes=False
):
    """Plot adjuster c(ℓ) and d(ℓ), optionally for every price realization."""
    _plot_adjuster_quantity(
        models, labels, t, title, f"{fname_prefix}_cAdj.pdf",
        "cFuncAdj_1d", "cFuncAdj_1d_list",
        r"Non-durable consumption, $c^{adj}(\ell_t)$",
        x_max=x_max, price_nodes=price_nodes,
    )
    _plot_adjuster_quantity(
        models, labels, t, title, f"{fname_prefix}_dAdj.pdf",
        "dFuncAdj_1d", "dFuncAdj_1d_list",
        r"Durable consumption, $d^{adj}(\ell_t)$",
        x_max=x_max, price_nodes=price_nodes,
    )


def plot_adjuster_mpcs(
    models, labels, t, title, fname_prefix, x_max=None, price_nodes=False
):
    """Plot adjuster MPCs, optionally for every price realization."""
    _plot_adjuster_quantity(
        models, labels, t, title, f"{fname_prefix}_cAdj_MPC.pdf",
        "cFuncAdj_1d", "cFuncAdj_1d_list",
        r"MPC non-durable, $\partial c^{adj}/\partial \ell_t$",
        x_max=x_max, price_nodes=price_nodes, derivative=True,
    )
    _plot_adjuster_quantity(
        models, labels, t, title, f"{fname_prefix}_dAdj_MPC.pdf",
        "dFuncAdj_1d", "dFuncAdj_1d_list",
        r"MPC durable, $\partial d^{adj}/\partial \ell_t$",
        x_max=x_max, price_nodes=price_nodes, derivative=True,
    )


def plot_adjuster_fues(models, labels, t, title, fname_prefix, x_max=None):
    """Scatter FOC candidates before/after FUES for adjuster c, d, and inv_v."""

    def _mask_x(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        if x_max is not None:
            keep = x <= x_max
            return x[keep], y[keep]
        return x, y

    def _inv_v(v, model):
        return np.nan_to_num(
            utility_inv(np.asarray(v, dtype=float), model.CRRA),
            nan=0.0, posinf=0.0, neginf=0.0,
        )

    def _scatter_one(ax, model, y_pre_attr, y_post_attr, transform=None):
        sol = model.solution[t]
        y_pre = getattr(sol, y_pre_attr)
        y_post = getattr(sol, y_post_attr)
        if transform is not None:
            y_pre = transform(y_pre, model)
            y_post = transform(y_post, model)
        x_pre, y_pre = _mask_x(sol.xAdj_pre, y_pre)
        x_post, y_post = _mask_x(sol.xAdj_post, y_post)
        ax.scatter(
            x_pre, y_pre, s=18, facecolors="none", edgecolors="k",
            alpha=0.55, linewidths=0.8, label=r"before FUES", zorder=2,
        )
        ax.scatter(
            x_post, y_post, s=22, color="k", marker=".",
            alpha=0.9, label=r"after FUES", zorder=3,
        )
        ax.set_xlabel(r"Liquidity, $\ell_t$")
        ax.set_xlim([x_pre[0], 8.0])
        handles, labs = ax.get_legend_handles_labels()
        _legend_under_title(ax, handles, labs)

    def _figure(y_pre_attr, y_post_attr, ylabel, ylabel_short, fname, transform=None):
        if len(models) == 1:
            fig, ax = plt.subplots(figsize=(6.5, 4.5))
            _scatter_one(ax, models[0], y_pre_attr, y_post_attr, transform)
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            fig.tight_layout()
        else:
            _n = len(models)
            _ncols = 2
            _nrows = int(np.ceil(_n / _ncols))
            fig, axes = plt.subplots(
                _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
            )
            axes = np.atleast_1d(axes).ravel()
            for ax, model, lab in zip(axes, models, labels):
                _scatter_one(ax, model, y_pre_attr, y_post_attr, transform)
                ax.set_title(lab)
                ax.set_ylabel(ylabel_short)
            for ax in axes[len(models) :]:
                ax.set_visible(False)
            fig.suptitle(title, y=0.98)
            fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.96])
        fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()

    _figure(
        "cAdj_pre", "cAdj_post",
        r"Non-durable consumption, $c^{adj}(\ell_t)$",
        r"$c^{adj}(\ell_t)$",
        f"{fname_prefix}_cAdj_FUES.pdf",
    )
    _figure(
        "dAdj_pre", "dAdj_post",
        r"Durable consumption, $d^{adj}(\ell_t)$",
        r"$d^{adj}(\ell_t)$",
        f"{fname_prefix}_dAdj_FUES.pdf",
    )
    _figure(
        "vAdj_pre", "vAdj_post",
        r"Inverse value, $\mathrm{inv}\,v^{adj}(\ell_t)$",
        r"$\mathrm{inv}\,v^{adj}(\ell_t)$",
        f"{fname_prefix}_inv_vAdj_FUES.pdf",
        transform=_inv_v,
    )


def plot_combined_policies(
    models,
    labels,
    title,
    fname_prefix,
    t=-1,
    n_vals=N_SHOW_DEFAULT,
    m_max=None,
):
    """Combined c(n, m) and d(n, m). One color per durable stock, all solid."""
    n_vals = tuple(n_vals)
    title_T = title + r", $t = T$"
    n_labels = [rf"$n={n0:g}$" for n0 in n_vals]

    def _m_grid(model):
        m = np.asarray(model.solution[t].mNrmGrid, dtype=float)
        if m_max is not None:
            m = m[m <= m_max]
        return m

    def _draw(ax, sol, m, which):
        handles = []
        for n0, color, lab in zip(n_vals, SWEEP_COLORS, n_labels):
            n_vec = np.full_like(m, float(n0), dtype=float)
            y = sol.cFunc(n_vec, m) if which == "c" else sol.dFunc(n_vec, m)
            h, = ax.plot(
                m, y, color=color, linestyle="-", linewidth=SWEEP_LW, label=lab,
            )
            handles.append(h)
        ax.set_xlabel(r"Market resources, $m_t$")
        return handles

    def _save(fig, handles, single, ylab, fname):
        if single:
            _legend_under_title(fig.axes[0], handles, n_labels)
            fig.tight_layout()
        else:
            fig.suptitle(title_T, y=1.12)
            _legend_under_suptitle(fig, handles, n_labels)
            fig.tight_layout()
        fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()

    # --- Non-durable ---
    if len(models) == 1:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        sol = models[0].solution[t]
        handles = _draw(ax, sol, _m_grid(models[0]), "c")
        ax.set_ylabel(r"Non-durable consumption, $c(n_t,m_t)$")
        ax.set_title(title_T)
        _save(fig, handles, True, None, f"{fname_prefix}_c_T.pdf")
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()
        handles = None
        for ax, model, lab in zip(axes, models, labels):
            handles = _draw(ax, model.solution[t], _m_grid(model), "c")
            ax.set_title(lab)
            ax.set_ylabel(r"$c(n_t,m_t)$")
        for ax in axes[len(models):]:
            ax.set_visible(False)
        _save(fig, handles, False, None, f"{fname_prefix}_c_T.pdf")

    # --- Durable ---
    if len(models) == 1:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        sol = models[0].solution[t]
        handles = _draw(ax, sol, _m_grid(models[0]), "d")
        ax.set_ylabel(r"Durable consumption, $d(n_t,m_t)$")
        ax.set_title(title_T)
        _save(fig, handles, True, None, f"{fname_prefix}_d_T.pdf")
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()
        handles = None
        for ax, model, lab in zip(axes, models, labels):
            handles = _draw(ax, model.solution[t], _m_grid(model), "d")
            ax.set_title(lab)
            ax.set_ylabel(r"$d(n_t,m_t)$")
        for ax in axes[len(models):]:
            ax.set_visible(False)
        _save(fig, handles, False, None, f"{fname_prefix}_d_T.pdf")


def plot_combined_mpcs(
    models,
    labels,
    title,
    fname_prefix,
    t=-1,
    n_vals=N_SHOW_DEFAULT,
    m_max=None,
):
    """Combined MPCs. One color per durable stock, all solid."""
    n_vals = tuple(n_vals)
    title_T = title + r", $t = T$"
    n_labels = [rf"$n={n0:g}$" for n0 in n_vals]

    def _m_grid(model):
        m = np.asarray(model.solution[t].mNrmGrid, dtype=float)
        if m_max is not None:
            m = m[m <= m_max]
        return m

    def _draw(ax, sol, m, which):
        handles = []
        for n0, color, lab in zip(n_vals, SWEEP_COLORS, n_labels):
            n_vec = np.full_like(m, float(n0), dtype=float)
            y = sol.cFunc(n_vec, m) if which == "c" else sol.dFunc(n_vec, m)
            y = np.asarray(y, dtype=float)
            h, = ax.plot(
                m, np.gradient(y, m),
                color=color, linestyle="-", linewidth=SWEEP_LW, label=lab,
            )
            handles.append(h)
        ax.set_xlabel(r"Market resources, $m_t$")
        return handles

    def _save(fig, handles, single, fname):
        if single:
            _legend_under_title(fig.axes[0], handles, n_labels)
        else:
            fig.suptitle(title_T, y=1.12)
            _legend_under_suptitle(fig, handles, n_labels)
        fig.tight_layout()
        fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()

    if len(models) == 1:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        handles = _draw(ax, models[0].solution[t], _m_grid(models[0]), "c")
        ax.set_ylabel(r"MPC non-durable, $\partial c/\partial m_t$")
        ax.set_title(title_T)
        _save(fig, handles, True, f"{fname_prefix}_c_T_MPC.pdf")
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()
        handles = None
        for ax, model, lab in zip(axes, models, labels):
            handles = _draw(ax, model.solution[t], _m_grid(model), "c")
            ax.set_title(lab)
            ax.set_ylabel(r"$\partial c/\partial m_t$")
        for ax in axes[len(models):]:
            ax.set_visible(False)
        _save(fig, handles, False, f"{fname_prefix}_c_T_MPC.pdf")

    if len(models) == 1:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        handles = _draw(ax, models[0].solution[t], _m_grid(models[0]), "d")
        ax.set_ylabel(r"MPC durable, $\partial d/\partial m_t$")
        ax.set_title(title_T)
        _save(fig, handles, True, f"{fname_prefix}_d_T_MPC.pdf")
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()
        handles = None
        for ax, model, lab in zip(axes, models, labels):
            handles = _draw(ax, model.solution[t], _m_grid(model), "d")
            ax.set_title(lab)
            ax.set_ylabel(r"$\partial d/\partial m_t$")
        for ax in axes[len(models):]:
            ax.set_visible(False)
        _save(fig, handles, False, f"{fname_prefix}_d_T_MPC.pdf")


def plot_combined_mpcs_decomp(
    models,
    labels,
    title,
    fname_prefix,
    t,
    n0=0.5,
    m_max=None,
):
    """
    MPC on the left (solid) and the level on the right (open markers).
    Color is the branch. Background is a gray/blue tint of the adjustment
    probability. With more than one price the adjuster is the node nearest 1.
    """
    from matplotlib.collections import PolyCollection
    from matplotlib.transforms import blended_transform_factory

    n_price = int(getattr(models[0], "PriceShkCount", 1) or 1)
    adj_name = r"adjuster, $p$ nearest 1" if n_price > 1 else "adjuster"
    branches = [
        ("cFunc", "dFunc", "combined", SWEEP_COLORS[0]),
        ("cFuncKeep", "dFuncKeep", "keeper", SWEEP_COLORS[1]),
        ("cFuncAdj", "dFuncAdj", adj_name, SWEEP_COLORS[2]),
    ]
    branch_labels = [b[2] for b in branches]

    def _m_grid(model, t_):
        m = np.asarray(model.solution[t_].mNrmGrid, dtype=float)
        if m_max is not None:
            m = m[m <= m_max]
        return m

    def _shade(ax, sol, m):
        n_vec = np.full_like(m, float(n0), dtype=float)
        adj = np.clip(np.asarray(sol.adjusting(n_vec, m), dtype=float), 0.0, 1.0)
        if m.size == 0:
            return
        if m.size == 1:
            edges = np.array([m[0] - 0.5, m[0] + 0.5])
        else:
            mid = 0.5 * (m[:-1] + m[1:])
            edges = np.empty(m.size + 1)
            edges[1:-1] = mid
            edges[0] = m[0] - (mid[0] - m[0])
            edges[-1] = m[-1] + (m[-1] - mid[-1])
        verts = [
            [(edges[i], 0.0), (edges[i + 1], 0.0), (edges[i + 1], 1.0), (edges[i], 1.0)]
            for i in range(m.size)
        ]
        keep = np.array(_KEEP_TINT)
        adj_c = np.array(_ADJ_TINT)
        rgb = (1.0 - adj)[:, None] * keep + adj[:, None] * adj_c
        colors = np.concatenate([rgb, np.ones((rgb.shape[0], 1))], axis=1)
        trans = blended_transform_factory(ax.transData, ax.transAxes)
        ax.add_collection(PolyCollection(
            verts, facecolors=colors, edgecolors="none", transform=trans, zorder=0,
        ))

    def _draw(ax, sol, m, which):
        _shade(ax, sol, m)
        n_vec = np.full_like(m, float(n0), dtype=float)
        ax2 = ax.twinx()
        handles = []
        step = max(1, int(m.size // 8))
        for c_attr, d_attr, lab, color in branches:
            func = getattr(sol, c_attr if which == "c" else d_attr)
            y = np.asarray(func(n_vec, m), dtype=float)
            h, = ax.plot(
                m, np.gradient(y, m),
                color=color, linestyle="-", linewidth=SWEEP_LW, label=lab, zorder=3,
            )
            ax2.plot(
                m, y, linestyle="None", marker="o",
                markerfacecolor="none", markeredgecolor=color,
                markeredgewidth=1.1, markersize=5.0, markevery=step, zorder=4,
            )
            handles.append(h)
        ax.set_xlabel(r"Market resources, $m_t$")
        ax.set_xlim([0, 5.0])
        ax.set_ylabel("MPC", color="k")
        ax2.set_ylabel("level", color="k")
        ax.tick_params(axis="y", labelcolor="k")
        ax2.tick_params(axis="y", labelcolor="k")
        return handles

    def _panel(t_, ttl, which, fname):
        if len(models) == 1:
            fig, ax = plt.subplots(figsize=(6.5, 4.5))
            handles = _draw(ax, models[0].solution[t_], _m_grid(models[0], t_), which)
            ax.set_title(ttl)
            _legend_under_title(ax, handles, branch_labels)
            fig.tight_layout()
        else:
            _n = len(models)
            _ncols = 2
            _nrows = int(np.ceil(_n / _ncols))
            fig, axes = plt.subplots(
                _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True,
            )
            axes = np.atleast_1d(axes).ravel()
            handles = None
            for ax, model, lab in zip(axes, models, labels):
                handles = _draw(ax, model.solution[t_], _m_grid(model, t_), which)
                ax.set_title(lab)
            for ax in axes[len(models):]:
                ax.set_visible(False)
            fig.suptitle(ttl, y=1.12)
            _legend_under_suptitle(fig, handles, branch_labels)
            fig.tight_layout()
        fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()

    def _horizon_tex(idx):
        n = len(models[0].solution)
        pos = idx if idx >= 0 else n + idx
        k = n - 1 - pos
        return r"T" if k == 0 else rf"T-{k}"

    for idx, tag in ((-1, "T"), (t, "periodt")):
        tex = _horizon_tex(idx)
        ttl = title + rf", $t = {tex}$, $n={n0:g}$"
        _panel(idx, ttl, "c", f"{fname_prefix}_c_{tag}_MPC_decomp.pdf")
        _panel(idx, ttl, "d", f"{fname_prefix}_d_{tag}_MPC_decomp.pdf")


def _heatmap_grid(models, panel_titles, t, fname, Z_list, cbar_label, vmin, vmax, cmap, suptitle=None):
    """Shared pcolormesh layout for Ss panels."""
    n = len(models)
    if n == 1:
        fig, axes = plt.subplots(1, 1, figsize=(6.0, 4.5))
        axes = np.array([axes])
    else:
        _n = len(models)
        _ncols = 2
        _nrows = int(np.ceil(_n / _ncols))
        fig, axes = plt.subplots(
            _nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), sharex=True, sharey=True,
        )
        axes = np.atleast_1d(axes).ravel()

    last_pcm = None
    for ax, model, ptitle, Z in zip(axes, models, panel_titles, Z_list):
        sol = model.solution[t]
        last_pcm = ax.pcolormesh(
            sol.mNrmGrid,
            sol.nNrmGrid,
            Z,
            cmap=cmap,
            shading="auto",
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(ptitle)
        ax.set_xlabel(r"Market resources, $m_t$")
        ax.set_ylabel(r"Durable stock, $n_t$")

    for ax in axes[n:]:
        ax.set_visible(False)

    if suptitle:
        fig.suptitle(suptitle, y=0.98)
    fig.tight_layout(rect=[0.0, 0.14, 1.0, 0.96 if suptitle else 1.0])
    cbar_ax = fig.add_axes([0.2, 0.04, 0.6, 0.03])
    fig.colorbar(last_pcm, cax=cbar_ax, orientation="horizontal", label=cbar_label)
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


def plot_ss_heatmaps(models, panel_titles, t, fname, suptitle=None, cbar_label="adjustment indicator"):
    """(S, s) heatmaps. Viridis, 0 to 1. The colorbar names the object in adjusting."""
    Z_list = [model.solution[t].adjusting_arr for model in models]
    _heatmap_grid(
        models,
        panel_titles,
        t,
        fname,
        Z_list,
        cbar_label=cbar_label,
        vmin=0.0,
        vmax=1.0,
        cmap="viridis",
        suptitle=suptitle,
    )


def plot_case_heatmaps(models, panel_titles, t, fname_stem, suptitle=None, cbar_label="adjustment indicator"):
    """Ss at period t and terminal."""
    plot_ss_heatmaps(
        models, panel_titles, t, f"{fname_stem}_Ss.pdf",
        suptitle=suptitle, cbar_label=cbar_label,
    )
    plot_ss_heatmaps(
        models, panel_titles, -1, f"{fname_stem}_Ss_T.pdf",
        suptitle=(suptitle + r" (terminal)") if suptitle else r"Terminal period",
        cbar_label=cbar_label,
    )


def solve_variant(base, label=None, **overrides):
    params = copy.deepcopy(base)
    params.update(overrides)
    model = DurableConsumer(**params)
    model.solve()
    model.print_kink_stats(label=label)
    return model


def collect_kink_euler_row(model, t, spec):
    """
    Period-``t`` kink counts plus off-grid Judd Euler residuals.

    Sample usage after any ``model.solve()``::

        model.compute_euler_errors()          # stores residuals on solution[t]
        sol = model.solution[t]
        sol.keep_decreasing_segments          # keeper DCEGM decreasing segments
        sol.adj_kinks_removed                 # adjuster FUES removals
        sol.ee_c_keep_mean, sol.ee_c_keep_p95
        sol.ee_d_adj_mean, sol.ee_d_adj_p95   # durable FOC in consumption units

    Residuals are branch-conditional (never the mixed policy) and use
    next period's stored marginal values. ``compute_euler_errors`` is called
    here if it has not been run yet.
    """
    sol = model.solution[t]
    if getattr(sol, "ee_c_keep_mean", None) is None:
        model.compute_euler_errors()
        sol = model.solution[t]
    return {
        "Specification": spec,
        "Decr. seg.": sol.keep_decreasing_segments,
        "Keep share": sol.keep_share_decreasing,
        "FUES rem.": sol.adj_kinks_removed,
        "Adj share": sol.adj_share_removed,
        "EE_c keep mean": sol.ee_c_keep_mean,
        "EE_c keep p95": sol.ee_c_keep_p95,
        "EE_c adj mean": sol.ee_c_adj_mean,
        "EE_c adj p95": sol.ee_c_adj_p95,
        "EE_d adj mean": sol.ee_d_adj_mean,
        "EE_d adj p95": sol.ee_d_adj_p95,
        "n keep": getattr(sol, "ee_c_keep_n", None),
        "n adj": getattr(sol, "ee_d_adj_n", None),
    }


def _fmt_int(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "---"
    return f"{int(x)}"


def _fmt_share(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "---"
    if abs(float(x)) < 1e-15:
        return "0"
    return f"{float(x):.3f}"


def _fmt_ee(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "---"
    return f"{float(x):.2f}"


def kink_euler_dataframe(rows):
    """Notebook display of kink counts and Euler residuals."""
    import pandas as pd

    cols = [
        "Specification",
        "Decr. seg.",
        "Keep share",
        "FUES rem.",
        "Adj share",
        "EE_c keep mean",
        "EE_c keep p95",
        "EE_c adj mean",
        "EE_c adj p95",
        "EE_d adj mean",
        "EE_d adj p95",
    ]
    df = pd.DataFrame(rows)[cols]
    for c in ("Decr. seg.", "FUES rem."):
        df[c] = df[c].map(_fmt_int)
    for c in ("Keep share", "Adj share"):
        df[c] = df[c].map(_fmt_share)
    for c in df.columns:
        if c.startswith("EE_"):
            df[c] = df[c].map(_fmt_ee)
    return df


def kink_euler_latex(rows, columns, caption, label, header_tex, colspec):
    """Build a tabular from ``collect_kink_euler_row`` dicts."""
    fmt = {
        "Specification": lambda x: x,
        "Decr. seg.": _fmt_int,
        "Keep share": _fmt_share,
        "FUES rem.": _fmt_int,
        "Adj share": _fmt_share,
        "EE_c keep mean": _fmt_ee,
        "EE_c keep p95": _fmt_ee,
        "EE_c adj mean": _fmt_ee,
        "EE_c adj p95": _fmt_ee,
        "EE_d adj mean": _fmt_ee,
        "EE_d adj p95": _fmt_ee,
    }
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\hline",
        header_tex,
        r"\hline",
    ]
    for row in rows:
        lines.append(" & ".join(fmt[c](row[c]) for c in columns) + r" \\")
    lines += [r"\hline", r"\end{tabular}", r"\end{table}", ""]
    return "\n".join(lines)


def print_kink_euler_table(rows, t, snapshot_specs=None, save_tex=True):
    """
    Print a pandas table and LaTeX for the paper.

    Writes ``../Text/smoothing_euler_table.tex`` (snapshot) and
    ``../Text/smoothing_euler_table_full.tex`` (full sweep) when ``save_tex``.
    """
    from IPython.display import display

    df = kink_euler_dataframe(rows)
    print(f"Period t = {t} (same index as the smoothing-shift figures)\n")
    display(df)

    full_cols = [
        "Specification",
        "Decr. seg.",
        "Keep share",
        "FUES rem.",
        "Adj share",
        "EE_c keep mean",
        "EE_c keep p95",
        "EE_d adj mean",
        "EE_d adj p95",
    ]
    full_header = (
        r" & \multicolumn{2}{c}{Keeper} & \multicolumn{2}{c}{Adjuster}"
        r" & \multicolumn{2}{c}{$\mathrm{EE}_c^{\mathrm{keep}}$}"
        r" & \multicolumn{2}{c}{$\mathrm{EE}_d^{\mathrm{adj}}$} \\"
        "\n"
        r"\cline{2-3}\cline{4-5}\cline{6-7}\cline{8-9}"
        "\n"
        r"Specification & Decr.\ seg. & Share & Kinks rem. & Share"
        r" & Mean & P95 & Mean & P95 \\"
    )
    full_tex = kink_euler_latex(
        rows,
        full_cols,
        caption=(
            r"Period-$t$ kink counts and off-grid Euler residuals along the "
            r"same four-channel sweep as Figure~\ref{fig:smoothing_shifts_levels}. "
            r"Residuals are mean and 95th percentile of~\eqref{eq:judd_ee}; "
            r"constrained points excluded. The $\sigma_p$ adjuster count sums "
            r"over price nodes."
        ),
        label="tab:smoothing_euler_full",
        header_tex=full_header,
        colspec="lcccccccc",
    )

    snap_cols = [
        "Specification",
        "Decr. seg.",
        "FUES rem.",
        "EE_c keep mean",
        "EE_c keep p95",
        "EE_d adj mean",
        "EE_d adj p95",
    ]
    snap_header = (
        r" & \multicolumn{2}{c}{Kinks}"
        r" & \multicolumn{2}{c}{$\mathrm{EE}_c^{\mathrm{keep}}$}"
        r" & \multicolumn{2}{c}{$\mathrm{EE}_d^{\mathrm{adj}}$} \\"
        "\n"
        r"\cline{2-3}\cline{4-5}\cline{6-7}"
        "\n"
        r"Specification & Decr.\ seg. & FUES rem. & Mean & P95 & Mean & P95 \\"
    )
    if snapshot_specs is None:
        snapshot_rows = rows
    else:
        wanted = set(snapshot_specs)
        snapshot_rows = [r for r in rows if r["Specification"] in wanted]
    snap_tex = kink_euler_latex(
        snapshot_rows,
        snap_cols,
        caption=(
            r"Period-$t$ kink counts and Euler residuals under the hard-max "
            r"baseline and one smoothed magnitude of each discrete-choice "
            r"channel. Residuals are mean and 95th percentile of~\eqref{eq:judd_ee} "
            r"on an off-grid test mesh; constrained points excluded. The "
            r"$\sigma_p$ adjuster kink count sums over price nodes."
        ),
        label="tab:smoothing_euler",
        header_tex=snap_header,
        colspec="lcccccc",
    )

    print("\n% ----- snapshot (paste into EulerError.tex as tab:smoothing_euler) -----")
    print(snap_tex)
    print("% ----- full sweep (tab:smoothing_euler_full) -----")
    print(full_tex)

    if save_tex:
        tex_dir = Path("../Text")
        snap_path = tex_dir / "smoothing_euler_table.tex"
        full_path = tex_dir / "smoothing_euler_table_full.tex"
        snap_path.write_text(snap_tex, encoding="utf-8")
        full_path.write_text(full_tex, encoding="utf-8")
        print("Wrote", snap_path.resolve())
        print("Wrote", full_path.resolve())
    return df


def plot_smoothing_policy_shifts(
    panels,
    t,
    n0=0.5,
    m_max=5.0,
    fname_prefix="SmoothingShifts",
):
    """
    Retirement-style comparison of combined policies under alternative smoothers.

    Produces four figures (shared axis limits within each figure):
      - combined c(n0, m) and d(n0, m)
      - MPCs dc/dm and dd/dm (numerical gradients)

    Parameters
    ----------
    panels : list of dict
        Four entries (a--d), each with keys ``title``, ``models``, ``labels``.
    t : int
        Period index to plot.
    n0 : float
        Fixed beginning-of-period durable stock.
    """
    assert len(panels) == 4, "expected four panels (a--d)"

    def _m_grid(model):
        m = np.asarray(model.solution[t].mNrmGrid, dtype=float)
        if m_max is not None:
            m = m[m <= m_max]
        return m

    def _series(panel, which, derivative=False):
        """Return list of (m, y) for every curve in a panel."""
        out = []
        for model in panel["models"]:
            sol = model.solution[t]
            m = _m_grid(model)
            n_vec = np.full_like(m, float(n0), dtype=float)
            y = sol.cFunc(n_vec, m) if which == "c" else sol.dFunc(n_vec, m)
            y = np.asarray(y, dtype=float)
            if derivative:
                y = np.gradient(y, m)
            out.append((m, y))
        return out

    def _draw_panel(ax, panel, series):
        handles = []
        for (m, y), lab, color in zip(series, panel["labels"], SWEEP_COLORS):
            h, = ax.plot(
                m, y, color=color, linestyle="-", linewidth=SWEEP_LW, label=lab,
            )
            handles.append(h)
        ax.set_xlabel(r"Market resources, $m_t$")
        ax.set_title(panel["title"])
        _legend_under_title(ax, handles, list(panel["labels"]))

    def _make_figure(which, ylab, sup, suffix, derivative=False):
        all_series = [_series(panel, which, derivative=derivative) for panel in panels]
        x_min = min(m.min() for series in all_series for m, _ in series)
        x_max = max(m.max() for series in all_series for m, _ in series)
        y_min = min(y.min() for series in all_series for _, y in series)
        y_max = max(y.max() for series in all_series for _, y in series)
        y_pad = 0.02 * (y_max - y_min) if y_max > y_min else 0.05

        fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
        axes = axes.ravel()
        for ax, panel, series in zip(axes, panels, all_series):
            _draw_panel(ax, panel, series)
            ax.set_ylabel(ylab)
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
        fig.suptitle(sup, y=1.02)
        fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.96])
        out = FIG_DIR / f"{fname_prefix}_{suffix}_periodt.pdf"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved", out.resolve())

    n_sol = len(panels[0]["models"][0].solution)
    pos = t if t >= 0 else n_sol + t
    k = n_sol - 1 - pos
    tex = "T" if k == 0 else (rf"T-{k}" if k > 0 else str(pos))

    _make_figure(
        "c",
        r"Non-durable consumption, $c_t(n_t,m_t)$",
        rf"Combined $c_t(n_t,m_t)$, $t={tex}$, $n={n0:g}$",
        "c",
    )
    _make_figure(
        "d",
        r"Durable stock, $d_t(n_t,m_t)$",
        rf"Combined $d_t(n_t,m_t)$, $t={tex}$, $n={n0:g}$",
        "d",
    )
    _make_figure(
        "c",
        r"MPC of non-durable consumption, $\partial c_t/\partial m_t$",
        rf"Combined MPC $\partial c_t/\partial m_t$, $t={tex}$, $n={n0:g}$",
        "c_MPC",
        derivative=True,
    )
    _make_figure(
        "d",
        r"MPC of durable stock, $\partial d_t/\partial m_t$",
        rf"Combined MPC $\partial d_t/\partial m_t$, $t={tex}$, $n={n0:g}$",
        "d_MPC",
        derivative=True,
    )


In [2]:
### Parameters for all figures
T = 5 #(T_cycle: number of periods before terminal)
t = 1 #(period to plot)

In [3]:
### Baseline parameters for all models

durable_defaults = dict(
    CRRA=2.0,
    DiscFac=0.975,
    LivPrb=1.0,
    alpha=0.75,
    d_ubar=1e-4,
    dDepr=0.05,
    adjC=0.10,
    MaintCost=0.0,
    OpCost=0.0,
    Rfree=1.04**(0.25),
    RSpread=0.0,
    PermGroFac=1.0,
    LTV=0.0,
    BoroCnstArt=0.0,
    # Income
    PermShkStd=0.0,
    PermShkCount=1,
    TranShkStd=0.2,
    TranShkCount=7,
    UnempPrb=0.0,
    IncUnemp=0.0,
    IncUnemp_Std=0.1,
    truncation=2,
    taxrate=0.0,
    labor=1.0,
    wage=1.0,
    # Channels (EV taste sigma_eps vs search are mutually exclusive)
    sigma_eps=0.0,
    kappa=0.0,
    SearchCost=0.1,
    ScaleBool=False,
    e_grid_count=101,
    PriceShkCount=5,
    PriceShkStd=0.1,
    # EoP method
    use_expected=True,
    use_TransitionMatrix=False,
    # Grids
    # Same asset and durable grids as the infinite-horizon notebooks.
    # cycles stays 1; those notebooks set cycles=0.
    nNrmMax=10,
    nNrmCount=200,
    mNrmMax=16.0,
    mNrmCount=200,
    xNrmMax=26,
    xNrmCount=100,
    aNrmMax=16.0,
    aNrmCount=200,
    NestFac=1,
    grid_type="exp_mult",
    TranMatrixmCount=250,
    TranMatrixmMax=16.0,
    TranMatrixnMax=10,
    TranMatrix_mCount=1010,
    TranMatrix_bCount=1000,
    N_scan=800,
    # Horizon: cycles=1 => finite lifecycle; T_cycle sets the number of periods.
    # (cycles=0 would instead solve the infinite horizon to convergence.)
    cycles=1,
    T_cycle=T,
    verbose=False,
)


### Baseline-model (hard max; no EV taste, search, or price)
Plot:
- non-durable consumption function of adjuster
- durable consumption function of the adjuster
- Ss structure (similar to DurableModel_One_Period.ipynb)
- Ss structure in the terminal period
- adjuster MPCs: $\partial c^{adj}/\partial \ell$ and $\partial d^{adj}/\partial \ell$
- combined (total) $c(n,m)$ and $d(n,m)$ at $t=T$ for $n\in\{0.5,1,1.5,2\}$
- combined MPCs: $\partial c/\partial m$ and $\partial d/\partial m$ at $t=T$ (same $n$)
- combined / adjuster(2D) / keeper MPCs and policies at $t=T$ and period $t$ for fixed $n=0.5$ (S,s shading)


In [4]:
Baseline_params = copy.deepcopy(durable_defaults)
Baseline_params["sigma_eps"] = 0.0
Baseline_params["SearchCost"] = 0.0
Baseline_params["PriceShkStd"] = 0.0
Baseline_params["PriceShkCount"] = 1

print("Solving baseline...")
t0 = time.time()
Baseline = DurableConsumer(**Baseline_params)
Baseline.solve()
Baseline.print_kink_stats(label="Baseline")
print(f"Baseline done in {time.time() - t0:.1f}s")


Solving baseline...


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)


=== Baseline: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 293, share of decreasing segments: 0.37229987293519695
  Adjuster — number of kinks removed: 60, share of removed points: 0.5172413793103449
Period 1:
  Keeper — number of decreasing segments: 431, share of decreasing segments: 0.40545625587958606
  Adjuster — number of kinks removed: 62, share of removed points: 0.5081967213114754
Period 2:
  Keeper — number of decreasing segments: 74, share of decreasing segments: 0.21203438395415472
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 285, share of decreasing segments: 0.36964980544747084
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 26, share of decreasing segments: 0.10276679841897234
  Adjuster — number of kinks removed: 0, share of removed points: nan
Period 5 (terminal): n/a
Baseline do

In [5]:
title = channel_title(0.0, 0.0, 0.0)
plot_adjuster_policies([Baseline], [r"baseline"], t, title, "Baseline")
plot_adjuster_mpcs([Baseline], [r"baseline"], t, title, "Baseline")
plot_adjuster_fues([Baseline], [r"baseline"], t, title, "Baseline")
plot_combined_policies([Baseline], [r"baseline"], title, "Baseline")
plot_combined_mpcs([Baseline], [r"baseline"], title, "Baseline")
plot_combined_mpcs_decomp([Baseline], [r"baseline"], title, "Baseline", t)
plot_case_heatmaps(
    [Baseline],
    [r"Baseline (hard max)"],
    t,
    "Baseline",
    suptitle=title,
    cbar_label="adjustment indicator",
)



C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:101: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:101: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:101: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:101: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:266: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:266: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:266: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:330: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:330: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:425: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:425: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
Baseline.solution

### Baseline-model (hard max; no EV taste, search, or price) with more shock points
- TranShkCount = 1, 7, 151
- TransitionMethod 

Plot:
- non-durable consumption function of adjuster (all 4 in one)
- durable consumption function of the adjuster (all 4 in one)
- Ss structure (similar to DurableModel_One_Period.ipynb) (4 subplots)

In [7]:
# Income discretization comparison (Figures_IncomeDiscMethod style):
# expected/Discretize with TranShkCount = 1, 7, 151 + TransitionMethod
base_inc = copy.deepcopy(durable_defaults)
base_inc["sigma_eps"] = 0.0
base_inc["SearchCost"] = 0.0
base_inc["PriceShkStd"] = 0.0
base_inc["PriceShkCount"] = 1

print("Solving TranShkCount=1 (expected)...")
Inc_1 = solve_variant(base_inc, label="Inc TranShkCount=1", TranShkCount=1, use_expected=True, use_TransitionMatrix=False)

print("Solving TranShkCount=7 (expected)...")
Inc_7 = solve_variant(base_inc, label="Inc TranShkCount=7", TranShkCount=7, use_expected=True, use_TransitionMatrix=False)

print("Solving TranShkCount=151 (expected)...")
Inc_151 = solve_variant(base_inc, label="Inc TranShkCount=151", TranShkCount=151, use_expected=True, use_TransitionMatrix=False)

print("Solving TransitionMethod...")
Inc_TM = solve_variant(
    base_inc,
    label="Inc TransitionMethod",
    TranShkCount=7,
    use_expected=False,
    use_TransitionMatrix=True,
)

inc_models = [Inc_1, Inc_7, Inc_151, Inc_TM]
inc_labels = [
    r"Discretize, $N=1$",
    r"Discretize, $N=7$",
    r"Discretize, $N=151$",
    r"TransitionMethod",
]
inc_panels = [
    r"Discretize, $N=1$",
    r"Discretize, $N=7$",
    r"Discretize, $N=151$",
    r"TransitionMethod",
]

title = channel_title(0.0, 0.0, 0.0, extra=r"income method")
plot_adjuster_policies(inc_models, inc_labels, t, title, "IncomeDisc")
plot_adjuster_mpcs(inc_models, inc_labels, t, title, "IncomeDisc")
plot_combined_policies(inc_models, inc_labels, title, "IncomeDisc")
plot_combined_mpcs(inc_models, inc_labels, title, "IncomeDisc")
plot_combined_mpcs_decomp(inc_models, inc_labels, title, "IncomeDisc", t)
plot_case_heatmaps(inc_models, inc_panels, t, "IncomeDisc", suptitle=title)



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


Solving TranShkCount=1 (expected)...


=== Inc TranShkCount=1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 568, share of decreasing segments: 0.42483171278982795
  Adjuster — number of kinks removed: 74, share of removed points: 0.5873015873015873
Period 1:
  Keeper — number of decreasing segments: 565, share of decreasing segments: 0.42449286250939144
  Adjuster — number of kinks removed: 78, share of removed points: 0.582089552238806
Period 2:
  Keeper — number of decreasing segments: 497, share of decreasing segments: 0.41589958158995816
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 60, share of decreasing segments: 0.18691588785046728
  Adjuster — number of kinks removed: 1, share of removed points: 0.014925373134328358
Period 4:
  Keeper — number of decreasing segments: 349, share of decreasing segments: 0.3882091212458287
  Adjuster — number of kinks removed: 0, share of removed points: nan
Period 5 (

=== Inc TranShkCount=7: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 293, share of decreasing segments: 0.37229987293519695
  Adjuster — number of kinks removed: 60, share of removed points: 0.5172413793103449
Period 1:
  Keeper — number of decreasing segments: 431, share of decreasing segments: 0.40545625587958606
  Adjuster — number of kinks removed: 62, share of removed points: 0.5081967213114754
Period 2:
  Keeper — number of decreasing segments: 74, share of decreasing segments: 0.21203438395415472
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 285, share of decreasing segments: 0.36964980544747084
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 26, share of decreasing segments: 0.10276679841897234
  Adjuster — number of kinks removed: 0, share of removed points: nan
Period 5 (terminal): n/a
S

=== Inc TranShkCount=151: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 5, share of decreasing segments: 0.023696682464454975
  Adjuster — number of kinks removed: 60, share of removed points: 0.5128205128205128
Period 1:
  Keeper — number of decreasing segments: 8, share of decreasing segments: 0.03686635944700461
  Adjuster — number of kinks removed: 56, share of removed points: 0.4827586206896552
Period 2:
  Keeper — number of decreasing segments: 10, share of decreasing segments: 0.04524886877828054
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 264, share of decreasing segments: 0.36213991769547327
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 27, share of decreasing segments: 0.10588235294117647
  Adjuster — number of kinks removed: 0, share of removed points: nan
Period 5 (terminal): n/a
So

...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### TasteShockModel (EV Type I; no search, no price)
- EV Type I taste scale $\sigma_\zeta \in \{0.1, 0.15, 0.2\}$

Plot:
- non-durable consumption function of adjuster (all 3 in one)
- durable consumption function of the adjuster (all 3 in one)
- adjuster FOC candidates before/after FUES (3 subplots)
- Ss structure (similar to DurableModel_One_Period.ipynb) (3 subplots)


In [8]:
base_taste = copy.deepcopy(durable_defaults)
base_taste["SearchCost"] = 0.0
base_taste["PriceShkStd"] = 0.0
base_taste["PriceShkCount"] = 1

sigmas = [0.1, 0.15, 0.2]
taste_models = []
for sx in sigmas:
    print(f"Solving taste sigma_eps={sx}...")
    taste_models.append(
        solve_variant(
            base_taste,
            label=f"taste sigma_eps={sx}",
            sigma_eps=sx,
        )
    )

taste_labels = [rf"$\sigma_{{\zeta}} = {sx}$" for sx in sigmas]
taste_panels = taste_labels
title = r"$\sigma_{\zeta} \in \{0.1, 0.15, 0.2\}$, $S = 0$, $\sigma_{p} = 0$"
plot_adjuster_policies(taste_models, taste_labels, t, title, "TasteShock")
plot_adjuster_mpcs(taste_models, taste_labels, t, title, "TasteShock")
plot_adjuster_fues(taste_models, taste_labels, t, title, "TasteShock")
plot_combined_policies(taste_models, taste_labels, title, "TasteShock")
plot_combined_mpcs(taste_models, taste_labels, title, "TasteShock")
plot_combined_mpcs_decomp(taste_models, taste_labels, title, "TasteShock", t)
plot_case_heatmaps(taste_models, taste_panels, t, "TasteShock", suptitle=title, cbar_label="choice probability")



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:184: RuntimeWarning: divide by zero encountered in power
  return uP ** (-1.0 / rho)


Solving taste sigma_eps=0.1...


=== taste sigma_eps=0.1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 6, share of removed points: 0.08108108108108109
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 5, share of removed points: 0.060240963855421686
Period 4:
  Keeper — number of decreasing segments: 76, share of decreasing segments: 0.21529745042492918
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving taste sigma_eps=0.15...


=== taste sigma_eps=0.15: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 58, share of decreasing segments: 0.1829652996845426
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving taste sigma_eps=0.2...


=== taste sigma_eps=0.2: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 50, share of decreasing segments: 0.16611295681063123
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 61, share of decreasing segments: 0.18885448916408668
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a


...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### SearchModel (no EV taste, no price)
- Search cost $S \in \{0.15, 0.2, 0.25\}$

Plot:
- non-durable consumption function of adjuster (all 3 in one)
- durable consumption function of the adjuster (all 3 in one)
- adjuster FOC candidates before/after FUES (3 subplots)
- Ss structure (similar to DurableModel_One_Period.ipynb) (3 subplots)


In [9]:
base_search = copy.deepcopy(durable_defaults)
base_search["sigma_eps"] = 0.0
base_search["PriceShkStd"] = 0.0
base_search["PriceShkCount"] = 1

S_vals = [0.15, 0.2, 0.25]
search_models = []
for S in S_vals:
    print(f"Solving SearchCost={S}...")
    search_models.append(solve_variant(base_search, label=f"SearchCost={S}", SearchCost=S))

search_labels = [rf"$S = {S}$" for S in S_vals]
title = r"$\sigma_{\zeta} = 0$, $S \in \{0.15, 0.2, 0.25\}$, $\sigma_{p} = 0$"
plot_adjuster_policies(search_models, search_labels, t, title, "Search")
plot_adjuster_mpcs(search_models, search_labels, t, title, "Search")
plot_adjuster_fues(search_models, search_labels, t, title, "Search")
plot_combined_policies(search_models, search_labels, title, "Search")
plot_combined_mpcs(search_models, search_labels, title, "Search")
plot_combined_mpcs_decomp(search_models, search_labels, title, "Search", t)
plot_case_heatmaps(search_models, search_labels, t, "Search", suptitle=title, cbar_label="search effort")



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


Solving SearchCost=0.15...


=== SearchCost=0.15: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 2, share of removed points: 0.03333333333333333
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 1, share of removed points: 0.016129032258064516
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 1, share of decreasing segments: 0.0049261083743842365
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving SearchCost=0.2...


=== SearchCost=0.2: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving SearchCost=0.25...


=== SearchCost=0.25: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a


...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### PriceModel (no EV taste, no search)
- Price standard deviation $\sigma_p \in \{0.05, 0.1, 0.15\}$

Plot:
- non-durable consumption function of adjuster (all 3 in one)
- durable consumption function of the adjuster (all 3 in one)
- Ss structure (similar to DurableModel_One_Period.ipynb) (3 subplots)


In [10]:
base_price = copy.deepcopy(durable_defaults)
base_price["sigma_eps"] = 0.0
base_price["SearchCost"] = 0.0

p_stds = [0.05, 0.1, 0.15]
price_models = []
for ps in p_stds:
    print(f"Solving PriceShkStd={ps}...")
    price_models.append(
        solve_variant(
            base_price,
            label=f"PriceShkStd={ps}",
            PriceShkStd=ps,
            PriceShkCount=1 if ps == 0.0 else base_price["PriceShkCount"],
        )
    )

price_labels = [rf"$\sigma_{{p}} = {ps}$" for ps in p_stds]
title = r"$\sigma_{\zeta} = 0$, $S = 0$, $\sigma_{p} \in \{0.05, 0.1, 0.15\}$"
plot_adjuster_policies(price_models, price_labels, t, title, "PriceStd", price_nodes=True)
plot_adjuster_mpcs(price_models, price_labels, t, title, "PriceStd", price_nodes=True)
plot_combined_policies(price_models, price_labels, title, "PriceStd")
plot_combined_mpcs(price_models, price_labels, title, "PriceStd")
plot_combined_mpcs_decomp(price_models, price_labels, title, "PriceStd", t)
plot_case_heatmaps(price_models, price_labels, t, "PriceStd", suptitle=title, cbar_label="probability of adjusting")



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


Solving PriceShkStd=0.05...


=== PriceShkStd=0.05: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 64, share of decreasing segments: 0.1945288753799392
  Adjuster — number of kinks removed: 172, share of removed points: 0.3747276688453159
Period 1:
  Keeper — number of decreasing segments: 92, share of decreasing segments: 0.23896103896103896
  Adjuster — number of kinks removed: 123, share of removed points: 0.2914691943127962
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 41, share of removed points: 0.11388888888888889
Period 3:
  Keeper — number of decreasing segments: 287, share of decreasing segments: 0.37032258064516127
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceShkStd=0

=== PriceShkStd=0.1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 43, share of decreasing segments: 0.14982578397212543
  Adjuster — number of kinks removed: 69, share of removed points: 0.19827586206896552
Period 1:
  Keeper — number of decreasing segments: 114, share of decreasing segments: 0.26573426573426573
  Adjuster — number of kinks removed: 112, share of removed points: 0.2679425837320574
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 78, share of removed points: 0.1989795918367347
Period 3:
  Keeper — number of decreasing segments: 422, share of decreasing segments: 0.40382775119617226
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceShkStd=0

=== PriceShkStd=0.15: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 21, share of decreasing segments: 0.08641975308641975
  Adjuster — number of kinks removed: 67, share of removed points: 0.1930835734870317
Period 1:
  Keeper — number of decreasing segments: 83, share of decreasing segments: 0.22615803814713897
  Adjuster — number of kinks removed: 59, share of removed points: 0.16954022988505746
Period 2:
  Keeper — number of decreasing segments: 2, share of decreasing segments: 0.00975609756097561
  Adjuster — number of kinks removed: 66, share of removed points: 0.17553191489361702
Period 3:
  Keeper — number of decreasing segments: 465, share of decreasing segments: 0.41114058355437666
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 164, share of removed points: 0.47674418604651164
Period 5 (te

...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Price (no EV taste, no search): varying PriceShkCount
- Price standard deviation = 0.1
- Price shock points = 1, 5, 13, 21

Plot:
- non-durable consumption function of adjuster (all 4 in one)
- durable consumption function of the adjuster (all 4 in one)
- Ss structure (similar to DurableModel_One_Period.ipynb) (4 subplots)

In [11]:
base_pcount = copy.deepcopy(durable_defaults)
base_pcount["sigma_eps"] = 0.0
base_pcount["SearchCost"] = 0.0
base_pcount["PriceShkStd"] = 0.1

p_counts = [1, 5, 13, 21]
pcount_models = []
for nc in p_counts:
    print(f"Solving PriceShkCount={nc}...")
    pcount_models.append(solve_variant(base_pcount, label=f"PriceShkCount={nc}", PriceShkCount=nc))

pcount_labels = [rf"$N_p = {nc}$" for nc in p_counts]
title = r"$\sigma_{\zeta} = 0$, $S = 0$, $\sigma_{p} = 0.1$"
plot_adjuster_policies(pcount_models, pcount_labels, t, title, "PriceCount", price_nodes=True)
plot_adjuster_mpcs(pcount_models, pcount_labels, t, title, "PriceCount", price_nodes=True)
plot_combined_policies(pcount_models, pcount_labels, title, "PriceCount")
plot_combined_mpcs(pcount_models, pcount_labels, title, "PriceCount")
plot_combined_mpcs_decomp(pcount_models, pcount_labels, title, "PriceCount", t)
plot_case_heatmaps(pcount_models, pcount_labels, t, "PriceCount", suptitle=title, cbar_label="probability of adjusting")

elapsed = time.time() - start_time
print("All durable figures done in", str(datetime.timedelta(seconds=elapsed)))
print("Saved under", FIG_DIR.resolve())



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


Solving PriceShkCount=1...


=== PriceShkCount=1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 293, share of decreasing segments: 0.37229987293519695
  Adjuster — number of kinks removed: 60, share of removed points: 0.5172413793103449
Period 1:
  Keeper — number of decreasing segments: 431, share of decreasing segments: 0.40545625587958606
  Adjuster — number of kinks removed: 62, share of removed points: 0.5081967213114754
Period 2:
  Keeper — number of decreasing segments: 74, share of decreasing segments: 0.21203438395415472
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 285, share of decreasing segments: 0.36964980544747084
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 26, share of decreasing segments: 0.10276679841897234
  Adjuster — number of kinks removed: 0, share of removed points: nan
Period 5 (terminal): n/a
Solv

=== PriceShkCount=5: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 43, share of decreasing segments: 0.14982578397212543
  Adjuster — number of kinks removed: 69, share of removed points: 0.19827586206896552
Period 1:
  Keeper — number of decreasing segments: 114, share of decreasing segments: 0.26573426573426573
  Adjuster — number of kinks removed: 112, share of removed points: 0.2679425837320574
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 78, share of removed points: 0.1989795918367347
Period 3:
  Keeper — number of decreasing segments: 422, share of decreasing segments: 0.40382775119617226
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceShkCount

=== PriceShkCount=13: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 14, share of decreasing segments: 0.0611353711790393
  Adjuster — number of kinks removed: 217, share of removed points: 0.22510373443983403
Period 1:
  Keeper — number of decreasing segments: 41, share of decreasing segments: 0.14487632508833923
  Adjuster — number of kinks removed: 255, share of removed points: 0.2463768115942029
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 152, share of removed points: 0.15816857440166493
Period 3:
  Keeper — number of decreasing segments: 317, share of decreasing segments: 0.37964071856287424
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceShkCou

...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


All durable figures done in 0:01:40.139598
Saved under C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Figures\DurableModel


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Price and Search Model (Combined; no EV taste)
- Search cost $S \in \{0.15, 0.2, 0.25\}$ at $\sigma_p = 0.1$
- Price dispersion $\sigma_p \in \{0.08, 0.1, 0.12\}$ at $S = 0.2$
- Price nodes = 9

Plot:
- non-durable consumption function of adjuster (all 5 in one)
- durable consumption function of the adjuster (all 5 in one)
- Ss structure (similar to DurableModel_One_Period.ipynb) (5 subplots)


In [12]:
base_price_search = copy.deepcopy(durable_defaults)
base_price_search["sigma_eps"] = 0.0
base_price_search["SearchCost"] = 0.0

price_search_specs = [
    (0.15, 0.1),
    (0.2, 0.08),
    (0.2, 0.1),
    (0.2, 0.12),
    (0.25, 0.1),
]
price_search_models = []
for S, ps in price_search_specs:
    print(f"Solving PriceSearch S={S}, sigma_p={ps}...")
    price_search_models.append(
        solve_variant(
            base_price_search,
            label=f"PriceSearch S={S} sigma_p={ps}",
            PriceShkStd=ps,
            PriceShkCount=9,
            SearchCost=S,
        )
    )

price_search_labels = [
    rf"$S = {S}$, $\sigma_{{p}} = {ps}$" for S, ps in price_search_specs
]
title = (
    r"$\sigma_{\zeta} = 0$, "
    r"$S \in \{0.15, 0.2, 0.25\}$ at $\sigma_{p} = 0.1$, "
    r"$\sigma_{p} \in \{0.08, 0.1, 0.12\}$ at $S = 0.2$"
)
plot_adjuster_policies(price_search_models, price_search_labels, t, title, "PriceStd_Search", price_nodes=True)
plot_adjuster_mpcs(price_search_models, price_search_labels, t, title, "PriceStd_Search", price_nodes=True)
plot_combined_policies(price_search_models, price_search_labels, title, "PriceStd_Search")
plot_combined_mpcs(price_search_models, price_search_labels, title, "PriceStd_Search")
plot_combined_mpcs_decomp(price_search_models, price_search_labels, title, "PriceStd_Search", t)
plot_case_heatmaps(price_search_models, price_search_labels, t, "PriceStd_Search", suptitle=title, cbar_label="probability of buying")



C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


Solving PriceSearch S=0.15, sigma_p=0.1...


=== PriceSearch S=0.15 sigma_p=0.1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 19, share of removed points: 0.03696498054474708
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 33, share of removed points: 0.05956678700361011
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 28, share of removed points: 0.04753820033955857
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 12, share of removed points: 0.019138755980861243
Period 4:
  Keeper — number of decreasing segments: 1, share of decreasing segments: 0.0049261083743842365
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceSearch S=

=== PriceSearch S=0.2 sigma_p=0.08: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 2, share of removed points: 0.0039447731755424065
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceSearch S=0.2, sigma_p=0.1...


=== PriceSearch S=0.2 sigma_p=0.1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 1, share of removed points: 0.002070393374741201
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 4, share of removed points: 0.00784313725490196
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 2, share of removed points: 0.0036496350364963502
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Solving PriceSearch S=0.2, sigma_p=0.12...


...

C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:584: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:639: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Smoothing channels: shifts in combined policies (period $t$)

Retirement-style comparison: all channels can smooth the keep--adjust discontinuity, but they shift the **combined** decision rules differently.

For fixed beginning-of-period durables $n_0=0.5$ and the notebook period index `t`, each panel overlays combined $c_t(n_0,m)$ / $d_t(n_0,m)$ for 3 magnitudes of one channel (others off, except income which holds a light EV taste scale so the discrete margin stays smooth):

- **(a)** EV taste $\sigma_\zeta \in \{0.1, 0.15, 0.2\}$
- **(b)** Search cost $S \in \{0.15, 0.2, 0.25\}$
- **(c)** Price dispersion $\sigma_p \in \{0.05, 0.1, 0.15\}$
- **(d)** Transitory income $\sigma_\eta \in \{0.1, 0.2, 0.4\}$ (with $\sigma_\zeta=0.1$)

Reference in each panel is the *smallest* listed magnitude (already a smoothed solution); larger values show the direction of the policy shift.

Four figures are saved: combined $c$ and $d$ levels, plus their MPCs ($\partial c/\partial m$, $\partial d/\partial m$).

The next cells also report, for the same models and period `t`, keeper decreasing-segment counts, adjuster FUES removals, and off-grid Euler residuals (`EE_c` keep / `EE_d` adj), and write a LaTeX table for the text.


In [13]:
# --- Smoothing-channel policy shifts (combined c, d at fixed n) ---
n0_smooth = 0.5  # same default as plot_combined_mpcs_decomp

smooth_base = copy.deepcopy(durable_defaults)
smooth_base["sigma_eps"] = 0.0
smooth_base["SearchCost"] = 0.0
smooth_base["PriceShkStd"] = 0.0
smooth_base["PriceShkCount"] = 1

# (a) EV taste
eps_vals = [0.1, 0.15, 0.2]
taste_shift_models = []
for sx in eps_vals:
    print(f"Smoothing shifts: sigma_eps={sx}...")
    taste_shift_models.append(
        solve_variant(
            smooth_base,
            label=f"SmoothShift sigma_eps={sx}",
            sigma_eps=sx,
            SearchCost=0.0,
            PriceShkStd=0.0,
            PriceShkCount=1,
        )
    )

# (b) Search cost
S_shift_vals = [0.15, 0.2, 0.25]
search_shift_models = []
for S in S_shift_vals:
    print(f"Smoothing shifts: SearchCost={S}...")
    search_shift_models.append(
        solve_variant(
            smooth_base,
            label=f"SmoothShift SearchCost={S}",
            sigma_eps=0.0,
            SearchCost=S,
            PriceShkStd=0.0,
            PriceShkCount=1,
        )
    )

# (c) Price dispersion
p_shift_vals = [0.05, 0.1, 0.15]
price_shift_models = []
for ps in p_shift_vals:
    print(f"Smoothing shifts: PriceShkStd={ps}...")
    price_shift_models.append(
        solve_variant(
            smooth_base,
            label=f"SmoothShift PriceShkStd={ps}",
            sigma_eps=0.0,
            SearchCost=0.0,
            PriceShkStd=ps,
            PriceShkCount=5,
        )
    )

# (d) Income shock size (light EV taste so keep--adjust is already smooth)
eta_vals = [0.1, 0.2, 0.4]
income_shift_models = []
for eta in eta_vals:
    print(f"Smoothing shifts: TranShkStd={eta}...")
    income_shift_models.append(
        solve_variant(
            smooth_base,
            label=f"SmoothShift TranShkStd={eta}",
            sigma_eps=0.1,
            SearchCost=0.0,
            PriceShkStd=0.0,
            PriceShkCount=1,
            TranShkStd=eta,
        )
    )

smoothing_panels = [
    dict(
        title=r"(a) Varying $\sigma_{\zeta}$",
        models=taste_shift_models,
        labels=[rf"$\sigma_{{\zeta}} = {sx}$" for sx in eps_vals],
    ),
    dict(
        title=r"(b) Varying $S$",
        models=search_shift_models,
        labels=[rf"$S = {S}$" for S in S_shift_vals],
    ),
    dict(
        title=r"(c) Varying $\sigma_{p}$",
        models=price_shift_models,
        labels=[rf"$\sigma_{{p}} = {ps}$" for ps in p_shift_vals],
    ),
    dict(
        title=r"(d) Varying $\sigma_{\eta}$",
        models=income_shift_models,
        labels=[rf"$\sigma_{{\eta}} = {eta}$" for eta in eta_vals],
    ),
]

plot_smoothing_policy_shifts(
    smoothing_panels, t, n0=n0_smooth, fname_prefix="SmoothingShifts"
)

elapsed = time.time() - start_time
print("All durable figures done in", str(datetime.timedelta(seconds=elapsed)))
print("Saved under", FIG_DIR.resolve())


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:650: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:657: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:184: RuntimeWarning: divide by zero encountered in power
  return uP ** (-1.0 / rho)


Smoothing shifts: sigma_eps=0.1...


=== SmoothShift sigma_eps=0.1: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 6, share of removed points: 0.08108108108108109
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 5, share of removed points: 0.060240963855421686
Period 4:
  Keeper — number of decreasing segments: 76, share of decreasing segments: 0.21529745042492918
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Smoothing shifts: sigma_eps=0.15...


=== SmoothShift sigma_eps=0.15: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 58, share of decreasing segments: 0.1829652996845426
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Smoothing shifts: sigma_eps=0.2...


=== SmoothShift sigma_eps=0.2: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 1:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 2:
  Keeper — number of decreasing segments: 0, share of decreasing segments: 0.0
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 3:
  Keeper — number of decreasing segments: 50, share of decreasing segments: 0.16611295681063123
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 4:
  Keeper — number of decreasing segments: 61, share of decreasing segments: 0.18885448916408668
  Adjuster — number of kinks removed: 0, share of removed points: 0.0
Period 5 (terminal): n/a
Smoothing shifts: SearchCost=0.15...


...

Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Figures\DurableModel\SmoothingShifts_d_periodt.pdf


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:977: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Figures\DurableModel\SmoothingShifts_c_MPC_periodt.pdf


Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Figures\DurableModel\SmoothingShifts_d_MPC_periodt.pdf
All durable figures done in 0:02:30.105199
Saved under C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Figures\DurableModel


C:\Users\amonninger\AppData\Local\Temp\ipykernel_5976\2752195858.py:977: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Kink counts and Euler residuals (period $t$)

For each solved model in the four-channel comparison (and the hard-max baseline), call `compute_euler_errors()` then read kinks and Judd residuals off `solution[t]`. Residuals are **branch-conditional** (keeper $c$, adjuster $d$-FOC in consumption units) on an off-grid $(n,m)$ mesh -- never the mixed policy $\bar c$.

Sample:

```python
model.compute_euler_errors()
sol = model.solution[t]
sol.keep_decreasing_segments, sol.adj_kinks_removed
sol.ee_c_keep_mean, sol.ee_c_keep_p95   # keeper consumption Euler
sol.ee_d_adj_mean, sol.ee_d_adj_p95     # adjuster durable Euler (consumption units)
model.print_kink_stats(label="example")
```

The code cell below runs that on every smoothing-shift model, prints a table, and writes LaTeX fragments `Text/smoothing_euler_table.tex` (snapshot matching Table `tab:smoothing_euler`) and `Text/smoothing_euler_table_full.tex` (full sweep).


In [14]:
# --- Sample: Euler residuals + kinks on the hard-max baseline ---
# After model.solve(), compute_euler_errors() fills solution[t] with
# Judd (1992) log10 relative residuals. print_kink_stats then shows both
# envelope counts and Euler mean / p95.

sol_base = Baseline.solution[t]
print("Sample call on Baseline, period t =", t)
print("  (kinks already stored at solve time; Euler computed next)")
Baseline.compute_euler_errors()
sol_base = Baseline.solution[t]
print("  Keeper decreasing segments:", sol_base.keep_decreasing_segments)
print("  Adjuster FUES removals:    ", sol_base.adj_kinks_removed)
print("  EE_c keep  mean / p95:     ", sol_base.ee_c_keep_mean, sol_base.ee_c_keep_p95)
print("  EE_d adj   mean / p95:     ", sol_base.ee_d_adj_mean, sol_base.ee_d_adj_p95)
Baseline.print_kink_stats(label="Baseline + Euler")

# --- Full table: same models as the four-panel smoothing figures ---
diag_rows = [
    collect_kink_euler_row(Baseline, t, r"Hard max"),
]
for sx, model in zip(eps_vals, taste_shift_models):
    diag_rows.append(
        collect_kink_euler_row(model, t, rf"$\sigma_{{\zeta}} = {sx}$")
    )
for S, model in zip(S_shift_vals, search_shift_models):
    diag_rows.append(
        collect_kink_euler_row(model, t, rf"$S = {S}$")
    )
for ps, model in zip(p_shift_vals, price_shift_models):
    diag_rows.append(
        collect_kink_euler_row(model, t, rf"$\sigma_{{p}} = {ps}$")
    )
for eta, model in zip(eta_vals, income_shift_models):
    diag_rows.append(
        collect_kink_euler_row(model, t, rf"$\sigma_{{\eta}} = {eta}$")
    )

snapshot_specs = [
    r"Hard max",
    r"$\sigma_{\zeta} = 0.05$",
    r"$S = 0.1$",
    r"$\sigma_{p} = 0.1$",
]
print_kink_euler_table(diag_rows, t, snapshot_specs=snapshot_specs)


Sample call on Baseline, period t = 1
  (kinks already stored at solve time; Euler computed next)


  Keeper decreasing segments: 431
  Adjuster FUES removals:     62
  EE_c keep  mean / p95:      -4.580180002573601 -2.3555862645898373
  EE_d adj   mean / p95:      -3.503688687692253 -2.2920457905579497
=== Baseline + Euler: kink / segment stats by period ===
Period 0:
  Keeper — number of decreasing segments: 293, share of decreasing segments: 0.37229987293519695
  Adjuster — number of kinks removed: 60, share of removed points: 0.5172413793103449
  Euler EE_c keep: mean=-4.503, p95=-2.375, n=108739, share>1e-3=0.105
  Euler EE_c adj: mean=-4.153, p95=-0.490, n=112055, share>1e-3=0.105
  Euler EE_d adj: mean=-3.414, p95=-2.100, n=104648, share>1e-3=0.198
Period 1:
  Keeper — number of decreasing segments: 431, share of decreasing segments: 0.40545625587958606
  Adjuster — number of kinks removed: 62, share of removed points: 0.5081967213114754
  Euler EE_c keep: mean=-4.580, p95=-2.356, n=107327, share>1e-3=0.106
  Euler EE_c adj: mean=-4.237, p95=-0.705, n=111995, share>1e-3=0.084


Period t = 1 (same index as the smoothing-shift figures)



,Specification,Decr. seg.,Keep share,FUES rem.,Adj share,EE_c keep mean,EE_c keep p95,EE_c adj mean,EE_c adj p95,EE_d adj mean,EE_d adj p95
0,Hard max,431,0.405,62,0.508,-4.58,-2.36,-4.24,-0.70,-3.50,-2.29
1,$\sigma_{\zeta} = 0.1$,0,0,0,0,-4.86,-4.02,-4.41,-0.73,-3.68,-0.33
2,$\sigma_{\zeta} = 0.15$,0,0,0,0,-4.99,-4.10,-4.33,-0.74,-3.49,-0.17
3,$\sigma_{\zeta} = 0.2$,0,0,0,0,-5.05,-4.12,-4.31,-0.75,-3.47,-0.11
4,$S = 0.15$,0,0,1,0.016,-4.70,-3.87,-4.19,-0.88,-3.12,-1.59
5,$S = 0.2$,0,0,0,0,-4.78,-3.97,-4.25,-0.85,-3.19,-2.10
6,$S = 0.25$,0,0,0,0,-4.86,-4.00,-4.35,-1.00,-3.42,-2.25
7,$\sigma_{p} = 0.05$,92,0.239,123,0.291,-4.34,-2.96,-4.27,-0.72,-3.34,-2.15
8,$\sigma_{p} = 0.1$,114,0.266,112,0.268,-4.35,-3.00,-4.36,-0.80,-3.46,-1.85
9,$\sigma_{p} = 0.15$,83,0.226,59,0.170,-4.35,-3.03,-4.28,-0.76,-2.98,-0.33



% ----- snapshot (paste into EulerError.tex as tab:smoothing_euler) -----
\begin{table}[htbp]
\centering
\caption{Period-$t$ kink counts and Euler residuals under the hard-max baseline and one smoothed magnitude of each discrete-choice channel. Residuals are mean and 95th percentile of~\eqref{eq:judd_ee} on an off-grid test mesh; constrained points excluded. The $\sigma_p$ adjuster kink count sums over price nodes.}
\label{tab:smoothing_euler}
\small
\begin{tabular}{lcccccc}
\hline
 & \multicolumn{2}{c}{Kinks} & \multicolumn{2}{c}{$\mathrm{EE}_c^{\mathrm{keep}}$} & \multicolumn{2}{c}{$\mathrm{EE}_d^{\mathrm{adj}}$} \\
\cline{2-3}\cline{4-5}\cline{6-7}
Specification & Decr.\ seg. & FUES rem. & Mean & P95 & Mean & P95 \\
\hline
Hard max & 431 & 62 & -4.58 & -2.36 & -3.50 & -2.29 \\
$\sigma_{p} = 0.1$ & 114 & 112 & -4.35 & -3.00 & -3.46 & -1.85 \\
\hline
\end{tabular}
\end{table}

% ----- full sweep (tab:smoothing_euler_full) -----
\begin{table}[htbp]
\centering
\caption{Period-$t$ kink 

,Specification,Decr. seg.,Keep share,FUES rem.,Adj share,EE_c keep mean,EE_c keep p95,EE_c adj mean,EE_c adj p95,EE_d adj mean,EE_d adj p95
0,Hard max,431,0.405,62,0.508,-4.58,-2.36,-4.24,-0.70,-3.50,-2.29
1,$\sigma_{\zeta} = 0.1$,0,0,0,0,-4.86,-4.02,-4.41,-0.73,-3.68,-0.33
2,$\sigma_{\zeta} = 0.15$,0,0,0,0,-4.99,-4.10,-4.33,-0.74,-3.49,-0.17
3,$\sigma_{\zeta} = 0.2$,0,0,0,0,-5.05,-4.12,-4.31,-0.75,-3.47,-0.11
4,$S = 0.15$,0,0,1,0.016,-4.70,-3.87,-4.19,-0.88,-3.12,-1.59
5,$S = 0.2$,0,0,0,0,-4.78,-3.97,-4.25,-0.85,-3.19,-2.10
6,$S = 0.25$,0,0,0,0,-4.86,-4.00,-4.35,-1.00,-3.42,-2.25
7,$\sigma_{p} = 0.05$,92,0.239,123,0.291,-4.34,-2.96,-4.27,-0.72,-3.34,-2.15
8,$\sigma_{p} = 0.1$,114,0.266,112,0.268,-4.35,-3.00,-4.36,-0.80,-3.46,-1.85
9,$\sigma_{p} = 0.15$,83,0.226,59,0.170,-4.35,-3.03,-4.28,-0.76,-2.98,-0.33
